# Day 18 – Quantisation – Extensive Solutions

Complete solutions for loading 4-bit models, comparing FP16 vs quantised, and using GGUF with llama.cpp.

In [ ]:
# !pip install transformers accelerate bitsandbytes torch

## Exercise 1: Compare FP16 vs 4-bit GPTQ vs GGUF Q4_K_M

We'll load the same model in different precisions and compare memory and output quality.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "gpt2"  # small model for demonstration; use "meta-llama/Llama-2-7b-hf" for real

# FP16
model_fp16 = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")
mem_fp16 = model_fp16.get_memory_footprint() / 1024**2
print(f"FP16 memory: {mem_fp16:.1f} MB")

# 4-bit with bitsandbytes
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
model_4bit = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quant_config, device_map="auto")
mem_4bit = model_4bit.get_memory_footprint() / 1024**2
print(f"4-bit memory: {mem_4bit:.1f} MB")
print(f"Memory reduction: {mem_fp16 / mem_4bit:.1f}x")

# Generate a sample to compare quality
tokenizer = AutoTokenizer.from_pretrained(model_name)
prompt = "The future of AI is"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")

out_fp16 = model_fp16.generate(**inputs, max_new_tokens=30)
out_4bit = model_4bit.generate(**inputs, max_new_tokens=30)
print("FP16 output:", tokenizer.decode(out_fp16[0]))
print("4-bit output:", tokenizer.decode(out_4bit[0]))

## Exercise 2: Quantise your own model (e.g., fine‑tuned GPT‑2) using optimum

We'll use `optimum` to quantise a model to INT8.

In [ ]:
# !pip install optimum
# from optimum.onnxruntime import ORTQuantizer
# from optimum.onnxruntime.configuration import AutoQuantizationConfig

# # Convert to ONNX first, then quantise
# quantizer = ORTQuantizer.from_pretrained("./merged_gpt2_cooking")
# dqconfig = AutoQuantizationConfig.avx512_vnni(is_static=False)
# quantizer.quantize(save_dir="./quantized_model", quantization_config=dqconfig)
print("Quantisation with optimum requires ONNX conversion – see documentation.")

## Exercise 3: CPU vs GPU speed comparison with GGUF

We'll use `llama-cpp-python` to run a GGUF model on CPU and compare with GPU (if available).

In [ ]:
# !pip install llama-cpp-python
from llama_cpp import Llama
import time

# Download a GGUF model (e.g., from Hugging Face)
# model_path = "path/to/llama-3-8b-Q4_K_M.gguf"
# llm = Llama(model_path=model_path, n_ctx=512, n_threads=4, n_gpu_layers=0)  # CPU only
# start = time.time()
# output = llm("What is AI?", max_tokens=50)
# elapsed = time.time() - start
# print(f"CPU speed: {len(output['choices'][0]['text'].split()) / elapsed:.1f} tok/sec")

# # With GPU offload (if CUDA available)
# llm_gpu = Llama(model_path=model_path, n_ctx=512, n_gpu_layers=35)  # offload most layers to GPU
# start = time.time()
# output_gpu = llm_gpu("What is AI?", max_tokens=50)
# elapsed_gpu = time.time() - start
# print(f"GPU speed: {len(output_gpu['choices'][0]['text'].split()) / elapsed_gpu:.1f} tok/sec")

print("To run this, download a GGUF model and adjust paths.")

## Exercise 4: Extreme quantisation – 2-bit

Try 2-bit quantisation and observe quality degradation.

In [ ]:
# For bitsandbytes, 2-bit is not directly supported; use 4-bit as lowest.
# For GGUF, you can use Q2_K (2-bit).
# Example with llama.cpp:
# llm_2bit = Llama(model_path="model-q2_k.gguf")
# out_2bit = llm_2bit("Explain gravity", max_tokens=50)
# print(out_2bit["choices"][0]["text"])
print("2-bit quantisation is available in GGUF format (Q2_K). Quality usually degrades significantly.")

## Bonus: Compare BERTScore between FP16 and 4-bit outputs

Quantitative evaluation of quality loss.

In [ ]:
from evaluate import load
bertscore = load("bertscore")

# Generate 10 responses from each model on the same prompts
prompts = ["Explain AI", "What is Python?", "Tell me a joke"]
fp16_outputs = []
bit4_outputs = []

# ... generate ... (pseudo)
# scores = bertscore.compute(predictions=bit4_outputs, references=fp16_outputs, lang="en")
# print(f"Average BERTScore F1: {sum(scores['f1'])/len(scores['f1']):.3f}")
print("Run generation loops to compare.")